In [ ]:
%pip install ipywidgets

In [1]:
import ipywidgets as widgets
from IPython.display import display
import tkinter as tk
from tkinter import filedialog
import os

# Global variable to store the path between cells
SELECTED_VIDEO_PATH = ""

def on_button_click(b):
    global SELECTED_VIDEO_PATH
    
    # Initialize tkinter
    root = tk.Tk()
    root.withdraw()
    root.attributes('-topmost', True) 
    
    # Open the dialog freely (no initialdir forced)
    file_path = filedialog.askopenfilename(
        title="Select any Video from your PC",
        filetypes=[("Video Files", "*.mp4 *.avi *.mov *.mkv"), ("All Files", "*.*")]
    )
    
    root.destroy()
    
    if file_path:
        SELECTED_VIDEO_PATH = file_path
        print(f"\n[SUCCESS] Video selected successfully!")
        print(f"Path: {SELECTED_VIDEO_PATH}")
        print("-> You can now run the next cell to process it.")
    else:
        print("\n[WARNING] Selection cancelled.")

# Create a visual button in the notebook
button = widgets.Button(
    description="Browse My Computer...",
    button_style="info", # Gives it a nice blue color
    icon="folder-open",
    layout=widgets.Layout(width='250px', height='40px')
)

button.on_click(on_button_click)

print("Click the button below to browse your directory and select a video:")
display(button)

Click the button below to browse your directory and select a video:


Button(button_style='info', description='Browse My Computer...', icon='folder-open', layout=Layout(height='40p…

In [19]:
# =========================================================
# MAGIC COMMANDS TO FORCE MODULE RELOAD
# =========================================================
%load_ext autoreload
%autoreload 2

import os
import cv2
import numpy as np
import joblib
import warnings
import sys
import traceback
from collections import deque

# Ignore scikit-learn warnings for cleaner output
warnings.filterwarnings("ignore", category=UserWarning)

# Import the feature extractor
from src.features.feature_extractor import extract_features_from_frame

# =========================================================
# 1. VERIFY SELECTION & SETUP PATHS
# =========================================================
if 'SELECTED_VIDEO_PATH' not in globals() or not SELECTED_VIDEO_PATH:
    print("[ERROR] No video selected! Please run the first cell.")
    sys.exit()

INPUT_VIDEO_PATH = SELECTED_VIDEO_PATH
original_filename = os.path.basename(INPUT_VIDEO_PATH)
name_only, extension = os.path.splitext(original_filename)

project_root = os.getcwd()

OUTPUT_DIR = os.path.join(project_root, "data", "processed_videos")
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_VIDEO_PATH = os.path.join(OUTPUT_DIR, f"test_output_{name_only}{extension}")

print(f"[INFO] Input Video: {original_filename}")
print(f"[INFO] Output Video will be saved as: {os.path.basename(OUTPUT_VIDEO_PATH)}")

# =========================================================
# 2. MODEL LOAD
# =========================================================
MODEL_PATH = "models/guitar_chord_rf_model.pkl"
print("[INFO] Loading Random Forest model...")
rf_model = joblib.load(MODEL_PATH)

# =========================================================
# 3. VIDEO PROCESSING SETUP
# =========================================================
cap = cv2.VideoCapture(INPUT_VIDEO_PATH)
if not cap.isOpened():
    print(f"[ERROR] Cannot open video {INPUT_VIDEO_PATH}")
    sys.exit()

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

if fps <= 0:
    fps = 30.0

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# --- MEMORY CONFIGURATION ---
# 0.5 seconds: Fast and responsive. The new model is strong enough 
# to not need the 1.0s "Tank Mode" delay anymore.
DESIRED_MEMORY_SECONDS = 0.5
SMOOTHING_WINDOW = max(5, int(fps * DESIRED_MEMORY_SECONDS))

print(f"[INFO] Processing video: {width}x{height} at {fps:.2f} FPS. Total frames: {total_frames}")

# Buffer to store recent predictions for democratic voting
prediction_buffer = deque(maxlen=SMOOTHING_WINDOW)

# Video writer setup
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, int(fps), (width, height))

# Cache and state variables
current_grid_cache = {}  
frame_count = 0

# =========================================================
# 4. MAIN INFERENCE LOOP
# =========================================================
while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1
    display_frame = frame.copy()

    try:
        # Extract features from the current frame
        features, current_grid_cache, debug_data = extract_features_from_frame(frame, current_grid_cache)

        if features is not None and len(features) == 38:  
            
            # Get probabilities from the Random Forest model
            probabilities = rf_model.predict_proba([features])[0]
            max_prob_index = np.argmax(probabilities)
            confidence = probabilities[max_prob_index]
            predicted_class = rf_model.classes_[max_prob_index]

            # =========================================================
            # THE "BOUNCER" THRESHOLD (Clean & Simple)
            # =========================================================
            # The new model is strong. We use a single, strict threshold to 
            # catch the 'N' transitions and kill false positives.
            strict_threshold = 0.20 

            # Apply the strict threshold
            if confidence < strict_threshold:
                raw_prediction = 'N'
            else:
                raw_prediction = predicted_class

            prediction_buffer.append(raw_prediction)

            # =========================================================
            # FAST DEMOCRATIC SMOOTHING (0.5 seconds)
            # =========================================================
            # Standard majority vote over the smaller, faster buffer
            if len(prediction_buffer) > 0:
                smoothed_prediction = max(set(prediction_buffer), key=prediction_buffer.count)
            else:
                smoothed_prediction = raw_prediction

            # =========================================================
            # GRID VISUALIZATION
            # =========================================================
            if debug_data is not None:
                matrix = debug_data.get('matrix')
                if matrix:
                    for string_nodes in matrix:
                        for i in range(len(string_nodes) - 1):
                            pt1 = string_nodes[i]
                            pt2 = string_nodes[i+1]
                            cv2.line(display_frame, pt1, pt2, (255, 0, 0), 2)
                            cv2.circle(display_frame, pt1, 4, (0, 165, 255), -1)
                        cv2.circle(display_frame, string_nodes[-1], 4, (0, 165, 255), -1)

                com_x = debug_data.get('com_x', 0)
                com_y = debug_data.get('com_y', 0)
                if com_y > 0 and matrix:
                    frets = debug_data.get('frets', [])
                    eqs = debug_data.get('eqs', [])
                    if frets and eqs:
                        viz_x = int(frets[int(min(com_x, len(frets)-1))])
                        viz_y = int(eqs[int(min(com_y, 5))][1])
                        cv2.circle(display_frame, (viz_x, viz_y), 12, (0, 255, 255), -1)
                        cv2.putText(display_frame, "COM", (viz_x + 15, viz_y), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

            # =========================================================
            # TEXT DRAWING
            # =========================================================
            text_color = (0, 255, 0) if smoothed_prediction != 'N' else (0, 0, 255)
            cv2.putText(display_frame, f"Chord: {smoothed_prediction}", (50, 100), cv2.FONT_HERSHEY_SIMPLEX, 2.5, text_color, 4)
            
            # Print debug info to screen
            debug_text = f"Raw AI: {raw_prediction} (Conf: {confidence:.2f} / Thr: {strict_threshold:.2f})"
            cv2.putText(display_frame, debug_text, (50, 160), cv2.FONT_HERSHEY_SIMPLEX, 1, (200, 200, 200), 2)
  
        else:
            cv2.putText(display_frame, "Tracking Lost", (50, 100), cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 0, 255), 4)
            # Instantly fill the buffer with 'N' to kill the lag
            for _ in range(SMOOTHING_WINDOW):
                prediction_buffer.append('N')

    except Exception as e:
        traceback.print_exc()
        error_msg = str(e)
        cv2.putText(display_frame, f"ERR: {error_msg}", (50, 100), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)

    out.write(display_frame)
    print(f"Processed frame {frame_count}/{total_frames}...", end='\r')

cap.release()
out.release()

print(f"\n[SUCCESS] Video saved as {os.path.basename(OUTPUT_VIDEO_PATH)}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
[INFO] Input Video: Test_video.MP4
[INFO] Output Video will be saved as: test_output_Test_video.MP4
[INFO] Loading Random Forest model...
[INFO] Processing video: 3840x2160 at 30.00 FPS. Total frames: 1690
Processed frame 1690/1690...
[SUCCESS] Video saved as test_output_Test_video.MP4
